In [10]:
# Create and validate our Spark session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("projet-final") \
        .master("local[*]") \
        .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.0


In [24]:
# Test the connection to the HDFS
hdfs_root_path = "hdfs://namenode:9000/user/mti850/"
file_name = "raw/olist_orders_dataset.csv"

from pyspark.sql.types import (
    StringType,
    StructField,
    StructType,
    TimestampType
)

schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_date", TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True),
])

df_raw = spark.read.format("csv") \
            .option("header", True) \
            .schema(schema) \
            .load(f"{hdfs_root_path}{file_name}")

df_raw.show(3)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [28]:
# Save the Dataframe in Parquet format
# Note: If you get an AccessControlException: Permission denied error 
# you'll need to give write permissions to the folder.

file_processed_name = "processed/olist_orders_dataset.parquet"

df_raw.write.mode("overwrite").option("compression", "snappy") \
    .parquet(f"{hdfs_root_path}{file_processed_name}")

# Commande: hdfs dfs -du -s -h /user/mti850/processed/olist_orders_dataset.parquet
#Taille sur disque (net)	Reduction
# CSV brut (Ligne)	/user/mti850/raw/olist_orders_dataset.csv	16.8 Mo
#Parquet + Snappy (Colonnaire)	/user/mti850/processed/olist_orders_dataset.parquet	6.4 Mo
# Reduction = 61.9 %

In [29]:
# Test de relecture
df_orders_parquet = spark.read. \
        parquet("hdfs://namenode:9000/user/mti850/processed/olist_orders_dataset.parquet")

df_orders_parquet.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [ ]:
spark.stop()